<a href="https://colab.research.google.com/github/pinarrcindemirr/DataScienceProject/blob/main/eco_travel_bot__.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installations

In [1]:
!sudo apt-get update

!sudo apt-get install python3.8 python3.8-venv -y

!python3.8 -m venv /usr/local/rasa_venv


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,560 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3.8 is already the newest version (3.8.20-1+

In [2]:
!source /usr/local/rasa_venv/bin/activate && pip install --upgrade pip

In [3]:
!source /usr/local/rasa_venv/bin/activate && \
pip install rasa==3.1.0 websockets==10.4 "sqlalchemy<2.0"

In [4]:
# Install NLTK and spaCy inside the venv
!source /usr/local/rasa_venv/bin/activate && pip install -q nltk

In [5]:
import nltk

# ===============================
# NLTK Resource Downloads
# ===============================

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

print('✓ NLTK resources ready.')

✓ NLTK resources ready.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


In [6]:
import os

PROJECT_DIR = "/content/eco_travel_bot"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/actions", exist_ok=True)

print("Eco Travel Bot project folder created.")

Eco Travel Bot project folder created.


In [7]:
%cd /content/eco_travel_bot

/content/eco_travel_bot


# config.yml

In [8]:
%%writefile /content/eco_travel_bot/config.yml

recipe: default.v1

language: en

# ────────────────────────────────────────────────────────────
# NLU PIPELINE — Pure DIETClassifier
# ────────────────────────────────────────────────────────────
pipeline:

  - name: WhitespaceTokenizer
  - name: RegexFeaturizer
  - name: LexicalSyntacticFeaturizer
  - name: CountVectorsFeaturizer
  - name: CountVectorsFeaturizer
    analyzer: char_wb
    min_ngram: 1
    max_ngram: 4

  - name: DIETClassifier
    epochs: 100
    constrain_similarities: true
    entity_recognition: true

  - name: EntitySynonymMapper

  - name: FallbackClassifier
    threshold: 0.70
    ambiguity_threshold: 0.10

  - name: RegexEntityExtractor
    use_lookup_tables: true
    use_regexes: true
    use_word_boundaries: true

# ────────────────────────────────────────────────────────────
# DIALOGUE POLICIES
# ────────────────────────────────────────────────────────────
policies:

  - name: MemoizationPolicy
    max_history: 8

  - name: RulePolicy
    core_fallback_threshold: 0.40
    core_fallback_action_name: "action_default_fallback"
    enable_fallback_prediction: true

  - name: TEDPolicy
    max_history: 10
    epochs: 100
    constrain_similarities: true

Overwriting /content/eco_travel_bot/config.yml


# Rasa core

## json/ mock.py

### hotels_raw.json

In [9]:
%%writefile /content/eco_travel_bot/data/hotels_raw.json
[
  {
    "id": "green_hostel",
    "name": "Green Hostel {city}",
    "tier": "budget",
    "eco_certified": true,
    "sustainability_score": 86,
    "amenities": ["shared rooms", "recycling program", "bike rental"]
  },
  {
    "id": "eco_pod",
    "name": "EcoPod {city}",
    "tier": "budget",
    "eco_certified": true,
    "sustainability_score": 82,
    "amenities": ["solar powered", "community kitchen", "recycling bins"]
  },
  {
    "id": "budget_inn",
    "name": "Budget Inn {city}",
    "tier": "budget",
    "eco_certified": false,
    "sustainability_score": 48,
    "amenities": ["basic amenities", "city centre location"]
  },
  {
    "id": "eco_stay",
    "name": "EcoStay {city}",
    "tier": "mid",
    "eco_certified": true,
    "sustainability_score": 91,
    "amenities": ["green certified", "electric vehicle charging", "organic breakfast", "bike rental"]
  },
  {
    "id": "green_hotel",
    "name": "Green Hotel {city}",
    "tier": "mid",
    "eco_certified": true,
    "sustainability_score": 84,
    "amenities": ["solar panels", "rainwater harvesting", "local produce restaurant"]
  },
  {
    "id": "city_boutique",
    "name": "City Boutique {city}",
    "tier": "mid",
    "eco_certified": false,
    "sustainability_score": 62,
    "amenities": ["rooftop terrace", "fitness centre", "city views"]
  },
  {
    "id": "standard_city",
    "name": "Standard City Hotel {city}",
    "tier": "mid",
    "eco_certified": false,
    "sustainability_score": 45,
    "amenities": ["business centre", "24h reception", "bar"]
  },
  {
    "id": "boutique_eco",
    "name": "Boutique Eco {city}",
    "tier": "high",
    "eco_certified": true,
    "sustainability_score": 94,
    "amenities": ["zero waste policy", "rooftop garden", "electric shuttle", "spa", "organic restaurant"]
  },
  {
    "id": "eco_luxury",
    "name": "EcoLuxury {city}",
    "tier": "high",
    "eco_certified": true,
    "sustainability_score": 90,
    "amenities": ["LEED certified", "carbon neutral", "heated pool", "concierge", "fine dining"]
  },
  {
    "id": "grand_green",
    "name": "Grand Green {city}",
    "tier": "high",
    "eco_certified": true,
    "sustainability_score": 88,
    "amenities": ["green roof", "private garden", "wellness centre", "electric car fleet"]
  },
  {
    "id": "premium_city",
    "name": "Premium City Hotel {city}",
    "tier": "high",
    "eco_certified": false,
    "sustainability_score": 55,
    "amenities": ["luxury spa", "michelin restaurant", "butler service", "rooftop bar"]
  }
]

Overwriting /content/eco_travel_bot/data/hotels_raw.json


### transport_profiles.json

In [10]:
%%writefile /content/eco_travel_bot/data/transport_profiles.json
{
  "train": {
    "carbon_per_100km": 6,
    "base_price_per_100km": 8,
    "eco_score": 95,
    "carbon_tier": "low",
    "carbon_label": "Green / Low emission",
    "price_tier": "medium",
    "description": "Most eco-friendly long-distance option"
  },
  "bus": {
    "carbon_per_100km": 10,
    "base_price_per_100km": 4,
    "eco_score": 88,
    "carbon_tier": "low",
    "carbon_label": "Green / Low emission",
    "price_tier": "low",
    "description": "Budget-friendly with low emissions"
  },
  "car": {
    "carbon_per_100km": 14,
    "base_price_per_100km": 12,
    "eco_score": 62,
    "carbon_tier": "medium",
    "carbon_label": "Amber / Moderate emission",
    "price_tier": "medium",
    "description": "Flexible but higher carbon footprint"
  },
  "flight": {
    "carbon_per_100km": 17,
    "base_price_per_100km": 18,
    "eco_score": 32,
    "carbon_tier": "high",
    "carbon_label": "Red / High emission",
    "price_tier": "high",
    "description": "Fastest but highest carbon footprint"
  }
}

Overwriting /content/eco_travel_bot/data/transport_profiles.json


### generate_mock_data.py

In [11]:
%%writefile /content/eco_travel_bot/generate_mock_data.py
import json
import os
import math

# ─────────────────────────────────────────────
# CONFIG: 18 şehir + Berlin'e km mesafesi
# ─────────────────────────────────────────────

CITY_CONFIG = {
    "berlin":     {"distance_km": 0,    "label": "Berlin"},
    "hamburg":    {"distance_km": 290,  "label": "Hamburg"},
    "munich":     {"distance_km": 585,  "label": "Munich"},
    "amsterdam":  {"distance_km": 650,  "label": "Amsterdam"},
    "prague":     {"distance_km": 350,  "label": "Prague"},
    "warsaw":     {"distance_km": 575,  "label": "Warsaw"},
    "vienna":     {"distance_km": 680,  "label": "Vienna"},
    "budapest":   {"distance_km": 870,  "label": "Budapest"},
    "paris":      {"distance_km": 1050, "label": "Paris"},
    "brussels":   {"distance_km": 780,  "label": "Brussels"},
    "copenhagen": {"distance_km": 380,  "label": "Copenhagen"},
    "rome":       {"distance_km": 1530, "label": "Rome"},
    "barcelona":  {"distance_km": 1760, "label": "Barcelona"},
    "madrid":     {"distance_km": 1870, "label": "Madrid"},
    "lisbon":     {"distance_km": 2310, "label": "Lisbon"},
    "london":     {"distance_km": 930,  "label": "London"},
    "istanbul":   {"distance_km": 1870, "label": "Istanbul"},
    "ankara":     {"distance_km": 2150, "label": "Ankara"},
    "izmir":      {"distance_km": 2220, "label": "Izmir"},
}

# Budget tier aralıkları (toplam paket = transport + 3 gece otel)
TIER_RANGES = {
    "budget": (100,  450),
    "mid":    (451,  750),
    "high":   (751, 9999),
}

# Her tier için uygun otel id'leri
TIER_HOTEL_IDS = {
    "budget": ["green_hostel", "eco_pod", "budget_inn"],
    "mid":    ["eco_stay", "green_hotel", "city_boutique", "standard_city"],
    "high":   ["boutique_eco", "eco_luxury", "grand_green", "premium_city"],
}

# Her tier'da hangi transport modları mantıklı
TIER_TRANSPORTS = {
    "budget": ["bus", "train"],
    "mid":    ["train", "car", "bus"],
    "high":   ["train", "flight", "car"],
}

# Minimum mesafe — kısa mesafelerde flight olmasın
FLIGHT_MIN_KM = 600


# ─────────────────────────────────────────────
# YARDIMCI FONKSİYONLAR
# ─────────────────────────────────────────────

def calc_carbon(transport_profile, distance_km):
    """Mesafeye göre carbon_kg hesapla, minimum 20 km (kısa şehirler için)."""
    km = max(distance_km, 20)
    raw = (transport_profile["carbon_per_100km"] / 100) * km
    return round(raw, 1)


def calc_transport_price(transport_profile, distance_km):
    """Mesafeye göre transport fiyatı (€), minimum 20 km."""
    km = max(distance_km, 20)
    raw = (transport_profile["base_price_per_100km"] / 100) * km
    # Gerçekçi tavan/taban
    return round(max(10, min(raw, 400)), 0)


def calc_hotel_price(tier, eco_certified, sustainability_score):
    """Tier + eco durumuna göre gecelik otel fiyatı (3 gece toplam)."""
    base = {"budget": 25, "mid": 150, "high": 260}[tier]
    # Eco sertifikalı oteller biraz daha pahalı
    eco_premium = 15 if eco_certified else 0
    # Yüksek sustainability score = biraz daha pahalı
    score_premium = round((sustainability_score - 50) / 10) * 5 if sustainability_score > 50 else 0
    per_night = base + eco_premium + score_premium
    return per_night * 3  # 3 gece


def get_carbon_tier(carbon_kg):
    if carbon_kg <= 60:  return "low"
    if carbon_kg <= 130: return "medium"
    return "high"


def get_carbon_label(carbon_kg):
    tier = get_carbon_tier(carbon_kg)
    return {
        "low":    "Green / Low emission",
        "medium": "Amber / Moderate emission",
        "high":   "Red / High emission",
    }[tier]


def get_budget_tier(total_price):
    if total_price <= 450: return "budget"
    if total_price <= 750: return "mid"
    return "high"


def hotel_name(template, city_label):
    return template.replace("{city}", city_label)


# ─────────────────────────────────────────────
# DOSYA YOLLARI
# ─────────────────────────────────────────────

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(BASE_DIR, "data")

with open(os.path.join(DATA_DIR, "hotels_raw.json"), encoding="utf-8") as f:
    HOTELS_RAW = json.load(f)

with open(os.path.join(DATA_DIR, "transport_profiles.json"), encoding="utf-8") as f:
    TRANSPORT_PROFILES = json.load(f)

# id → hotel dict için index
HOTEL_BY_ID = {h["id"]: h for h in HOTELS_RAW}


# ─────────────────────────────────────────────
# ANA ÜRETIM DÖNGÜSÜ
# ─────────────────────────────────────────────

travel_options = {}

for city_key, city_info in CITY_CONFIG.items():
    distance  = city_info["distance_km"]
    city_label = city_info["label"]

    # Her tier için seçenekleri biriktir
    tier_buckets = {"budget": [], "mid": [], "high": []}

    for transport_name, t_profile in TRANSPORT_PROFILES.items():

        # Kısa mesafede flight olmasın
        if transport_name == "flight" and distance < FLIGHT_MIN_KM:
            continue
        # Berlin'e 0 km mesafe → minimum uygulanır, sadece "local" modlar
        if distance == 0 and transport_name == "flight":
            continue

        carbon_kg        = calc_carbon(t_profile, distance)
        transport_price  = calc_transport_price(t_profile, distance)
        carbon_tier_val  = get_carbon_tier(carbon_kg)
        carbon_label_val = get_carbon_label(carbon_kg)

        # Bu transport için tier'a uygun otelleri eşleştir
        for tier_name, hotel_ids in TIER_HOTEL_IDS.items():

            # Flight sadece mid ve high'ta
            if transport_name == "flight" and tier_name == "budget":
                continue
            # Bus sadece budget ve mid'de
            if transport_name == "bus" and tier_name == "high":
                continue

            for hotel_id in hotel_ids:
                hotel = HOTEL_BY_ID[hotel_id]
                if hotel["tier"] != tier_name:
                    continue

                hotel_price = calc_hotel_price(tier_name, hotel["eco_certified"], hotel["sustainability_score"])
                total_price = int(transport_price + hotel_price)

                # Gerçek tier'ı fiyata göre belirle (overlap olabilir)
                actual_tier = get_budget_tier(total_price)

                record = {
                    "transport":            transport_name,
                    "hotel":                hotel_name(hotel["name"], city_label),
                    "hotel_tier":           tier_name,
                    "eco_certified":        hotel["eco_certified"],
                    "price":                total_price,
                    "carbon_kg":            carbon_kg,
                    "carbon_tier":          carbon_tier_val,
                    "carbon_label":         carbon_label_val,
                    "sustainability_score": hotel["sustainability_score"],
                    "amenities":            hotel["amenities"],
                }

                tier_buckets[actual_tier].append(record)

    # Her tier'da duplicate transport+hotel kombinasyonlarını temizle
    # ve en iyi 4 seçeneği bırak (sustainability_score'a göre sırala)
    for tier_name in tier_buckets:
        seen = set()
        unique = []
        for rec in tier_buckets[tier_name]:
            key = (rec["transport"], rec["hotel"])
            if key not in seen:
                seen.add(key)
                unique.append(rec)
        # En iyi 4: önce eco_certified, sonra sustainability_score
        unique.sort(key=lambda x: (x["eco_certified"], x["sustainability_score"]), reverse=True)
        tier_buckets[tier_name] = unique[:4]

    travel_options[city_key] = tier_buckets


# ─────────────────────────────────────────────
# OUTPUT
# ─────────────────────────────────────────────

output = {
    "travel_options":             travel_options,
    "transport_carbon_profiles":  TRANSPORT_PROFILES,
}

out_path = os.path.join(DATA_DIR, "mock_travel_data.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

# Özet
total_records = sum(
    len(tiers[t])
    for tiers in travel_options.values()
    for t in tiers
)
print(f"✓ mock_travel_data.json generated")
print(f"  Cities  : {len(travel_options)}")
print(f"  Records : {total_records} total options across all cities and tiers")
for city, tiers in list(travel_options.items())[:3]:
    print(f"  {city}: budget={len(tiers['budget'])} mid={len(tiers['mid'])} high={len(tiers['high'])}")
print("  ...")

Overwriting /content/eco_travel_bot/generate_mock_data.py


### local_activities.json

In [12]:
%%writefile /content/eco_travel_bot/data/local_activities.json
{
  "berlin": {
    "high":   ["Brandenburg Gate eco walking tour", "Cycling along the Berlin Wall trail", "Tiergarten nature hike"],
    "medium": ["Spree river guided walk", "Cycling through Prenzlauer Berg", "Tempelhof field open-air walk"],
    "low":    ["Berlin TV Tower sightseeing", "Hop-on hop-off bus city tour", "Alexanderplatz neighbourhood stroll"]
  },
  "hamburg": {
    "high":   ["Alster lake nature walk", "Cycling through Alster parklands", "Planten un Blomen garden walk"],
    "medium": ["Speicherstadt canal walking tour", "Cycling through HafenCity", "Blankenese riverside stroll"],
    "low":    ["Harbour boat sightseeing tour", "Hop-on hop-off bus city tour", "Reeperbahn neighbourhood walk"]
  },
  "paris": {
    "high":   ["Seine riverside eco walking tour", "Cycling in Bois de Boulogne", "Montmartre neighbourhood hike"],
    "medium": ["Le Marais district walking tour", "Cycling along Canal Saint-Martin", "Luxembourg Gardens stroll"],
    "low":    ["Eiffel Tower sightseeing tour", "Seine river cruise", "Champs-Élysées shopping walk"]
  },
  "amsterdam": {
    "high":   ["Canal district eco walking tour", "Cycling through Vondelpark", "Jordaan neighbourhood food walk"],
    "medium": ["Museumplein walking tour", "Cycling through Westerpark", "De Pijp neighbourhood stroll"],
    "low":    ["Canal boat sightseeing tour", "Hop-on hop-off bus city tour", "Rijksmuseum neighbourhood walk"]
  },
  "rome": {
    "high":   ["Colosseum area eco walking tour", "Cycling in Villa Borghese park", "Trastevere neighbourhood walk"],
    "medium": ["Campo de Fiori market walk", "Cycling along the Appian Way", "Pigneto neighbourhood stroll"],
    "low":    ["Vatican sightseeing tour", "Hop-on hop-off bus city tour", "Trevi Fountain neighbourhood walk"]
  },
  "vienna": {
    "high":   ["Ringstrasse eco walking tour", "Cycling along the Danube canal", "Prater park nature walk"],
    "medium": ["Naschmarkt food walking tour", "Cycling through Augarten park", "Spittelberg neighbourhood stroll"],
    "low":    ["Schönbrunn Palace sightseeing", "Hop-on hop-off bus city tour", "Graben shopping walk"]
  },
  "barcelona": {
    "high":   ["Gothic Quarter eco walking tour", "Cycling along Barceloneta beach", "Park Güell nature hike"],
    "medium": ["El Born district walking tour", "Cycling through Ciutadella park", "Gràcia neighbourhood stroll"],
    "low":    ["Sagrada Familia sightseeing tour", "Hop-on hop-off bus city tour", "Las Ramblas shopping walk"]
  },
  "copenhagen": {
    "high":   ["Nyhavn harbour eco walking tour", "Cycling through Frederiksberg gardens", "Christiania neighbourhood walk"],
    "medium": ["Vesterbro district walking tour", "Cycling through Fælledparken", "Torvehallerne food market walk"],
    "low":    ["Tivoli Gardens sightseeing", "Hop-on hop-off bus city tour", "Strøget shopping walk"]
  },
  "london": {
    "high":   ["Thames riverside eco walking tour", "Cycling in Hyde Park", "Borough Market food walk"],
    "medium": ["Southbank cultural walking tour", "Cycling through Victoria Park", "Shoreditch neighbourhood stroll"],
    "low":    ["Tower of London sightseeing", "Hop-on hop-off bus city tour", "Oxford Street shopping walk"]
  },
  "lisbon": {
    "high":   ["Alfama district eco walking tour", "Cycling along the Tagus river", "Sintra hills nature hike"],
    "medium": ["Bairro Alto walking tour", "Cycling through Monsanto park", "LX Factory neighbourhood stroll"],
    "low":    ["Belém Tower sightseeing tour", "Hop-on hop-off bus city tour", "Chiado shopping walk"]
  },
  "madrid": {
    "high":   ["Retiro Park eco walking tour", "Cycling through Casa de Campo", "La Latina neighbourhood food walk"],
    "medium": ["Malasaña district walking tour", "Cycling along the Manzanares river", "Chueca neighbourhood stroll"],
    "low":    ["Royal Palace sightseeing tour", "Hop-on hop-off bus city tour", "Gran Vía shopping walk"]
  },
  "istanbul": {
    "high":   ["Sultanahmet eco walking tour", "Cycling along the Bosphorus", "Balat neighbourhood culture walk"],
    "medium": ["Karaköy district walking tour", "Cycling through Belgrad forest", "Kadıköy food market walk"],
    "low":    ["Topkapi Palace sightseeing tour", "Bosphorus boat cruise", "Grand Bazaar shopping walk"]
  },
  "munich": {
    "high":   ["Englischer Garten eco walking tour", "Cycling along the Isar river", "Marienplatz neighbourhood walk"],
    "medium": ["Schwabing district walking tour", "Cycling through Olympiapark", "Viktualienmarkt food walk"],
    "low":    ["Nymphenburg Palace sightseeing", "Hop-on hop-off bus city tour", "Kaufingerstrasse shopping walk"]
  },
  "prague": {
    "high":   ["Old Town eco walking tour", "Cycling through Stromovka park", "Vinohrady neighbourhood walk"],
    "medium": ["Malá Strana district walking tour", "Cycling along the Vltava river", "Žižkov neighbourhood stroll"],
    "low":    ["Prague Castle sightseeing tour", "Hop-on hop-off bus city tour", "Wenceslas Square shopping walk"]
  },
  "warsaw": {
    "high":   ["Old Town eco walking tour", "Cycling along the Vistula river", "Lazienki Park nature walk"],
    "medium": ["Praga district walking tour", "Cycling through Kabaty forest", "Żoliborz neighbourhood stroll"],
    "low":    ["Palace of Culture sightseeing", "Hop-on hop-off bus city tour", "Nowy Świat shopping walk"]
  },
  "budapest": {
    "high":   ["Castle Hill eco walking tour", "Cycling along the Danube promenade", "Margaret Island nature walk"],
    "medium": ["Jewish Quarter walking tour", "Cycling through City Park", "Kazinczy street food walk"],
    "low":    ["Parliament sightseeing tour", "Danube river cruise", "Váci Street shopping walk"]
  },
  "ankara": {
    "high":   ["Hamamonu district eco walking tour", "Cycling in Gençlik Park", "Atakule neighbourhood walk"],
    "medium": ["Ulus district walking tour", "Cycling through Eymir lake trail", "Kızılay neighbourhood stroll"],
    "low":    ["Anıtkabir sightseeing tour", "Hop-on hop-off bus city tour", "Kızılay shopping walk"]
  },
  "izmir": {
    "high":   ["Kordon seafront eco walking tour", "Cycling through Kültürpark", "Kemeraltı bazaar food walk"],
    "medium": ["Alsancak district walking tour", "Cycling along the coastal path", "Bornova neighbourhood stroll"],
    "low":    ["Kadifekale sightseeing tour", "Hop-on hop-off bus city tour", "Kemeralti shopping walk"]
  },
  "default": {
    "high":   ["City eco walking tour", "Local cycling route", "Nature park hike"],
    "medium": ["City centre walking tour", "Neighbourhood cycling route", "Local food market walk"],
    "low":    ["City sightseeing tour", "Hop-on hop-off bus tour", "Shopping district walk"]
  }
}

Overwriting /content/eco_travel_bot/data/local_activities.json


## domain

In [13]:
%%writefile /content/eco_travel_bot/domain.yml
version: "3.1"

intents:
  - compare_transport
  - provide_transport_mode_compare
  - goodbye
  - ask_privacy
  - ask_accessibility
  - ask_ethics
  - plan_trip
  - provide_destination
  - provide_origin
  - provide_date
  - provide_budget
  - provide_preference
  - provide_location_method
  - provide_transport_mode
  - show_local_activities
  - provide_option
  - affirm
  - deny
  - request_human
  - out_of_scope
  - nlu_fallback

entities:
  - destination
  - origin
  - date
  - budget
  - preference
  - location_method
  - transport_mode

slots:
  destination:
    type: text
    influence_conversation: true
    mappings:
      - type: custom

  origin:
    type: text
    influence_conversation: true
    mappings:
      - type: custom

  date:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: date

  budget:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: budget

  preference:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: preference

  location_method:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: location_method

  transport_mode:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: transport_mode

  clarification_count:
    type: float
    initial_value: 0
    influence_conversation: false
    mappings:
      - type: custom

  api_error_count:
    type: float
    initial_value: 0
    influence_conversation: false
    mappings:
      - type: custom

  handover_reason:
    type: text
    influence_conversation: true
    mappings:
      - type: custom

  requested_slot:
    type: text
    influence_conversation: true
    mappings:
      - type: custom

  selected_option:
    type: text
    influence_conversation: true
    mappings:
      - type: custom

responses:
  utter_welcome:
    - text: "Hello! Welcome to Eco-Travel Advisor \U0001F33F I can help you plan sustainable trips, compare transport options, or connect you with a human travel advisor. How would you like to start?"
      buttons:
        - title: "Plan a sustainable trip"
          payload: "/plan_trip"
        - title: "Compare transport options"
          payload: "/compare_transport"
        - title: "Talk to a human advisor"
          payload: "/request_human"

  utter_goodbye:
    - text: "Thank you for using Eco-Travel Advisor \U0001F33F Have a sustainable trip!"

  utter_privacy_info:
    - text: "Your privacy matters. This chatbot only uses your trip details to generate sustainable travel recommendations. It does not permanently store personal travel data. If you request human handover, your trip context will be shared with the advisor in JSON format."

  utter_ask_destination:
    - text: "Where would you like to travel?"

  utter_ask_origin:
    - text: "Where are you traveling from?"

  utter_ask_date:
    - text: "When are you planning to travel?"

  utter_ask_budget:
    - text: "What is your approximate budget? You can choose an option or type a custom budget."
      buttons:
        - title: "Low"
          payload: '/provide_budget{"budget":"low"}'
        - title: "Medium"
          payload: '/provide_budget{"budget":"medium"}'
        - title: "High"
          payload: '/provide_budget{"budget":"high"}'

  utter_ask_preference:
    - text: "What level of sustainability do you prefer?"
      buttons:
        - title: "Low"
          payload: '/provide_preference{"preference":"low"}'
        - title: "Medium"
          payload: '/provide_preference{"preference":"medium"}'
        - title: "High"
          payload: '/provide_preference{"preference":"high"}'

  utter_ask_location_method:
    - text: "Where are you traveling from? You can type your departure city or use GPS."
      buttons:
        - title: "Use GPS"
          payload: '/provide_location_method{"location_method":"gps"}'

  utter_ask_transport_mode:
    - text: "Which transport mode do you prefer?"
      buttons:
        - title: "Car"
          payload: '/provide_transport_mode{"transport_mode":"car"}'
        - title: "Bus"
          payload: '/provide_transport_mode{"transport_mode":"bus"}'
        - title: "Flight"
          payload: '/provide_transport_mode{"transport_mode":"flight"}'
        - title: "Train"
          payload: '/provide_transport_mode{"transport_mode":"train"}'
        - title: "No preference"
          payload: '/provide_transport_mode{"transport_mode":"any"}'

  utter_ask_missing_info:
    - text: "I need a bit more information to continue. Could you please provide the missing details?"

  utter_accessibility:
    - text: "This chatbot supports accessibility through clear language, simple navigation, quick-reply buttons, and screen-reader-friendly responses."

  utter_ethics:
    - text: "I provide transparent sustainability guidance. Carbon scores are approximate, and I avoid unsupported environmental claims or greenwashing."

  utter_fallback:
    - text: "Sorry, I didn't understand that. Can you rephrase?"
      buttons:
        - title: "Plan a sustainable trip"
          payload: "/plan_trip"
        - title: "Compare transport options"
          payload: "/compare_transport"
        - title: "Talk to human advisor"
          payload: "/request_human"

  utter_ask_rephrase:
    - text: "Could you rephrase that in another way?"

  utter_recommendations:
    - text: "Here are some sustainable travel options for you."

  utter_out_of_scope:
    - text: "I can only help with sustainable travel planning, eco-friendly transport, accommodation recommendations, local activities, carbon footprint information, and travel-related support."

  utter_handover:
    - text: "I will connect you to a human travel advisor."

actions:
  - action_ask_destination
  - action_set_destination
  - action_call_travel_api
  - action_calculate_carbon
  - action_rank_options
  - action_check_missing_slots
  - action_ask_origin
  - action_set_origin
  - action_compare_transport_options
  - action_confirm_compare_selection
  - action_show_local_activities
  - action_set_selected_option
  - action_handover
  - action_retry_or_escalate
  - action_default_fallback

session_config:
  session_expiration_time: 60
  carry_over_slots_to_new_session: true

Overwriting /content/eco_travel_bot/domain.yml


## nlu

In [14]:
%%writefile /content/eco_travel_bot/data/nlu.yml
version: "3.1"

nlu:

- intent: goodbye
  examples: |
    - bye
    - goodbye
    - see you
    - thanks bye
    - thank you goodbye
    - end conversation
    - finish

- intent: plan_trip
  examples: |
    - I want to plan a trip
    - I want to travel
    - Help me plan a trip
    - I need a travel plan
    - Plan a trip for me
    - I need help with my trip
    - I want travel advice
    - Recommend me a trip
    - Suggest a travel plan
    - Create a travel itinerary for me

    - I want an eco-friendly trip
    - Help me plan an eco-friendly trip
    - I want a sustainable trip
    - Plan a sustainable trip for me
    - I need a low carbon trip
    - I want a green travel plan
    - I want to travel sustainably
    - Suggest a low emission travel option

    - I want to travel to [Berlin](destination)
    - I want to travel [Berlin](destination)
    - Travel to [Amsterdam](destination)
    - Travel [Amsterdam](destination)
    - Trip to [Paris](destination)
    - Plan a trip to [Rome](destination)
    - Help me plan a trip to [Barcelona](destination)
    - I want to go to [Lisbon](destination)
    - I would like to visit [Madrid](destination)
    - Check [Vienna](destination)

    - I want to travel from [Berlin](origin) to [Hamburg](destination)
    - I want to travel from [Amsterdam](origin) to [Berlin](destination)
    - Plan a trip from [Paris](origin) to [Rome](destination)
    - Help me plan a trip from [Madrid](origin) to [Lisbon](destination)
    - I need a travel plan from [Barcelona](origin) to [Vienna](destination)
    - I want to go from [Rome](origin) to [Paris](destination)

    - I want to travel [Berlin](origin) to [Hamburg](destination)
    - I want to travel [Amsterdam](origin) to [Berlin](destination)
    - I want to travel [Berlin](origin) to [Amsterdam](destination)
    - I want to travel [London](origin) to [Paris](destination)
    - I want to travel [Rome](origin) to [Vienna](destination)
    - travel [Berlin](origin) to [Amsterdam](destination)
    - travel [Paris](origin) to [Rome](destination)
    - trip [Madrid](origin) to [Lisbon](destination)
    - go [Berlin](origin) to [Amsterdam](destination)
    - [Berlin](origin) to [Amsterdam](destination)
    - [Paris](origin) to [London](destination)
    - [Rome](origin) to [Vienna](destination)

    - I want an eco-friendly trip to [Berlin](destination)
    - Plan a sustainable trip to [Amsterdam](destination)
    - I need a low carbon trip to [Paris](destination)
    - Help me plan a green trip to [Rome](destination)
    - I want to travel sustainably to [Barcelona](destination)

    - I want a sustainable trip from [Berlin](origin) to [Hamburg](destination)
    - I want an eco-friendly trip from [Amsterdam](origin) to [Berlin](destination)
    - Plan a low carbon trip from [Berlin](origin) to [Paris](destination)
    - Help me plan a green trip from [Paris](origin) to [Rome](destination)

    - I want to travel from [Berlin](origin) to [Amsterdam](destination) by [train](transport_mode)
    - Plan a trip from [Amsterdam](origin) to [Berlin](destination) by [bus](transport_mode)
    - I want a low carbon trip from [Berlin](origin) to [Paris](destination) by [train](transport_mode)
    - Help me plan an eco-friendly trip from [Paris](origin) to [Rome](destination) by [bus](transport_mode)

    - I want to travel to [Berlin](destination) next week
    - Plan a trip to [Amsterdam](destination) tomorrow
    - Help me plan a trip to [Paris](destination) next month
    - I want an eco-friendly trip to [Rome](destination) next weekend

    - I want to travel to [Berlin](destination) under [100 euros](budget)
    - Plan a trip to [Amsterdam](destination) with a budget of [200 euros](budget)
    - I need a sustainable trip to [Paris](destination) under [300 euros](budget)

    - I want a low carbon trip from [Berlin](origin) to [Paris](destination) by [train](transport_mode) next week under [100 euros](budget)
    - Plan an eco-friendly trip from [Amsterdam](origin) to [Berlin](destination) by [bus](transport_mode) [tomorrow](date) with a budget of [150 euros](budget)

- intent: compare_transport
  examples: |
    - compare transport options
    - I want to compare transport options
    - compare train and flight
    - compare bus and train
    - show me transport options
    - which transport is more eco friendly
    - compare carbon emissions
    - compare travel modes
    - help me choose transport
    - compare car and train
    - compare flight and bus
    - what is the best transport option
    - show low carbon transport options
    - which transport has lower emissions

- intent: provide_transport_mode_compare
  examples: |
    - [train](transport_mode)
    - [bus](transport_mode)
    - [flight](transport_mode)
    - [car](transport_mode)


- intent: provide_origin
  examples: |
    - from [Berlin](origin)
    - from [London](origin)
    - from [Paris](origin)
    - from [Rome](origin)
    - from [Vienna](origin)
    - from [Istanbul](origin)
    - from [Amsterdam](origin)
    - from [Munich](origin)
    - from [Hamburg](origin)
    - from [Barcelona](origin)
    - from [Copenhagen](origin)
    - from [Lisbon](origin)
    - from [Madrid](origin)
    - from [Prague](origin)
    - from [Warsaw](origin)
    - from [Budapest](origin)
    - from [Ankara](origin)
    - from [Izmir](origin)

    - I am traveling from [Berlin](origin)
    - I am travelling from [Amsterdam](origin)
    - I will travel from [Hamburg](origin)
    - I will start from [Paris](origin)
    - I want to start from [Copenhagen](origin)
    - I am starting from [Ankara](origin)
    - I am coming from [Amsterdam](origin)
    - My starting point is [Rome](origin)
    - My departure city is [Istanbul](origin)
    - My departure point is [Izmir](origin)
    - My trip starts in [Ankara](origin)
    - The origin is [Hamburg](origin)
    - My city is [Amsterdam](origin)
    - Leaving from [Munich](origin)
    - My departure is [Paris](origin)
    - I'm leaving from [Paris](origin)
    - Starting from [Paris](origin)
    - Coming from [Paris](origin)
    - I'm from [Berlin](origin)
    - departing from [Vienna](origin)
    - departure city is [Rome](origin)

- intent: provide_destination
  examples: |
    - [Berlin](destination)
    - [Hamburg](destination)
    - [Paris](destination)
    - [Rome](destination)
    - [Amsterdam](destination)
    - [Vienna](destination)
    - [Barcelona](destination)
    - [Copenhagen](destination)
    - [London](destination)
    - [Lisbon](destination)
    - [Madrid](destination)
    - [Istanbul](destination)
    - [Munich](destination)
    - [Ankara](destination)
    - [Izmir](destination)
    - [Prague](destination)
    - [Warsaw](destination)
    - [Budapest](destination)
    - to [Berlin](destination)
    - to [Paris](destination)
    - to [Amsterdam](destination)
    - to [Copenhagen](destination)
    - to [Barcelona](destination)
    - to [Hamburg](destination)
    - to [Rome](destination)
    - to [Vienna](destination)
    - to [London](destination)
    - to [Lisbon](destination)
    - to [Madrid](destination)
    - to [Istanbul](destination)
    - to [Munich](destination)
    - to [Prague](destination)
    - to [Warsaw](destination)
    - to [Budapest](destination)

    - I want to go to [Paris](destination)
    - I want to travel to [Amsterdam](destination)
    - I would like to visit [Copenhagen](destination)
    - I would like to go to [London](destination)
    - I am going to [Barcelona](destination)
    - I am travelling to [Berlin](destination)
    - My destination is [Amsterdam](destination)
    - My trip destination is [Berlin](destination)
    - Destination is [Rome](destination)
    - The destination is [Hamburg](destination)
    - I want to visit [Izmir](destination)
    - travelling to [Vienna](destination)
    - going to [Prague](destination)
    - heading to [Budapest](destination)

- intent: provide_date
  examples: |
    - [today](date)
    - [tomorrow](date)
    - [next week](date)
    - [next month](date)
    - today
    - tomorrow
    - I want to travel on [June 10](date)
    - The date is [next week](date)
    - I am planning to travel [tomorrow](date)
    - I want to go in [July](date)
    - My travel date is [May 20](date)
    - I plan to travel [this weekend](date)
    - I want to travel [in August](date)
    - The trip is planned for [June 15](date)
    - [this weekend](date)
    - [in August](date)
    - [July](date)
    - [next Saturday](date)

- intent: provide_budget
  examples: |
    - My budget is [1000 euros](budget)
    - I have [800 euros](budget)
    - Around [500 euros](budget)
    - [1200 euros](budget)
    - My budget is [600 euros](budget)
    - I can spend [700 euros](budget)
    - The budget is [1500 euros](budget)
    - under [1000 euros](budget)
    - less than [900 euros](budget)
    - [1500](budget)
    - [1000](budget)
    - [500](budget)
    - [2000](budget)
    - [750](budget)
    - [300](budget)
    - [100](budget)
    - [3000](budget)
    - [1500€](budget)
    - [500€](budget)
    - [1000€](budget)
    - [1500 euro](budget)
    - [500 euro](budget)
    - [800 eur](budget)
    - budget is [1500](budget)
    - around [500](budget)
    - about [1200](budget)
    - I have [300](budget)
    - [low](budget)
    - [medium](budget)
    - [high](budget)
    - low budget
    - medium budget
    - high budget
    - I want a cheap trip
    - I have a flexible budget
    - cheap
    - affordable
    - expensive
    - premium budget

- intent: provide_preference
  examples: |
    - [low](preference)
    - [medium](preference)
    - [high](preference)
    - I prefer [low](preference) sustainability
    - I want [medium](preference) sustainability
    - I prefer [high](preference) eco options
    - sustainability level [high](preference)
    - choose [medium](preference)
    - I want the most eco-friendly option
    - I want a balanced option
    - sustainability is very important to me
    - I do not care much about sustainability
    - I prefer eco-friendly recommendations
    - I want green travel options
    - I want sustainable travel
    - eco friendly please
    - green transport
    - low carbon option

- intent: provide_location_method
  examples: |
    - Use [GPS](location_method)
    - Please use my [gps](location_method)
    - Detect my location using [GPS](location_method)
    - [gps](location_method)
    - use my current location
    - detect my location
    - use GPS please

- intent: provide_transport_mode
  examples: |
    - [car](transport_mode)
    - [bus](transport_mode)
    - [flight](transport_mode)
    - [train](transport_mode)
    - [any](transport_mode)
    - I prefer [car](transport_mode)
    - I prefer [bus](transport_mode)
    - I prefer [flight](transport_mode)
    - I prefer [train](transport_mode)
    - I want to travel by [car](transport_mode)
    - I want to travel by [bus](transport_mode)
    - I want to travel by [flight](transport_mode)
    - I want to travel by [train](transport_mode)
    - I have [no preference](transport_mode)
    - No preference
    - any transport is fine
    - I do not have a transport preference
    - I do not mind
    - whatever is best
    - choose the best option for me

- intent: show_local_activities
  examples: |
    - show local activities
    - show me local activities
    - what can I do locally
    - recommend local activities
    - show walking tours
    - show cycling routes
    - show hiking experiences
    - I want local experiences
    - suggest cultural experiences
    - what activities can I do there

- intent: provide_option
  examples: |
    - option 1
    - option 2
    - option 3
    - option 4
    - I choose option 1
    - I choose option 2
    - I select option 3
    - I want option 1
    - I pick option 2
    - first option
    - second option
    - third option
    - fourth option
    - 1
    - 2
    - 3
    - 4

- intent: ask_privacy
  examples: |
    - privacy information
    - do you store my data
    - what happens to my data
    - is my data stored
    - how do you handle my personal data
    - GDPR
    - data privacy
    - do you keep my personal information
    - will my data be shared
    - what personal data do you use

- intent: ask_accessibility
  examples: |
    - is this chatbot accessible
    - do you support accessibility
    - accessibility support
    - can screen readers use this chatbot
    - is it screen-reader friendly
    - do you support disabled users
    - how do you make the chatbot accessible

- intent: ask_ethics
  examples: |
    - is this ethical
    - how do you avoid greenwashing
    - are your sustainability claims reliable
    - are carbon scores accurate
    - how do you handle ethical issues
    - can I trust your eco recommendations
    - do you make unsupported environmental claims

- intent: affirm
  examples: |
    - yes
    - correct
    - exactly
    - sure
    - that is right
    - okay
    - yes please
    - confirmed
    - sounds good

- intent: deny
  examples: |
    - no
    - not really
    - I don't want that
    - incorrect
    - no thanks
    - that is wrong
    - I do not agree
    - not this one

- intent: request_human
  examples: |
    - I want to talk to a human
    - Connect me to an advisor
    - I need a travel specialist
    - Human advisor please
    - Can I speak with someone?
    - I need expert help
    - transfer me to a human advisor
    - I want human support
    - this is too complex, I need an advisor

- intent: out_of_scope
  examples: |
    - hello
    - hi
    - hey
    - good morning
    - order pizza
    - can you order food
    - play music
    - tell me a joke
    - buy iphone
    - book dentist
    - what is the weather
    - send an email
    - open spotify
    - who won the football match
    - write my homework
    - make a restaurant reservation
    - book a doctor appointment

- intent: nlu_fallback
  examples: |
    - asdfgh
    - I don't know
    - maybe something
    - what is this
    - random message

- regex: budget
  examples: |
    - \b\d{2,6}(\s*(euros?|eur|€))?\b

Overwriting /content/eco_travel_bot/data/nlu.yml


## stories

In [15]:
%%writefile /content/eco_travel_bot/data/stories.yml
version: "3.1"

stories:

- story: basic multi turn eco travel planning
  steps:
    - intent: plan_trip
    - action: action_check_missing_slots
    - intent: provide_destination
    - action: action_check_missing_slots
    - intent: provide_origin
    - action: action_check_missing_slots
    - intent: provide_date
    - action: action_check_missing_slots
    - intent: provide_budget
    - action: action_check_missing_slots
    - intent: provide_preference
    - action: action_check_missing_slots
    - intent: provide_transport_mode
    - action: action_check_missing_slots

- story: smart flexible input with complete message
  steps:
    - intent: plan_trip
    - action: action_check_missing_slots

- story: compare transport options flow
  steps:
    - intent: compare_transport
    - action: action_compare_transport_options

- story: show local activities flow
  steps:
    - intent: show_local_activities
    - action: action_show_local_activities

- story: local activities after recommendations
  steps:
    - action: action_rank_options
    - intent: affirm
    - action: action_show_local_activities

- story: skip local activities after recommendations
  steps:
    - action: action_rank_options
    - intent: deny
    - action: utter_goodbye

- story: select option then local activities
  steps:
    - action: action_rank_options
    - intent: provide_option
    - action: action_set_selected_option
    - intent: affirm
    - action: action_show_local_activities

- story: select option then no local activities
  steps:
    - action: action_rank_options
    - intent: provide_option
    - action: action_set_selected_option
    - intent: deny
    - action: utter_goodbye

- story: privacy question flow
  steps:
    - intent: ask_privacy
    - action: utter_privacy_info

- story: accessibility question flow
  steps:
    - intent: ask_accessibility
    - action: utter_accessibility

- story: ethics question flow
  steps:
    - intent: ask_ethics
    - action: utter_ethics

- story: human handover requested directly
  steps:
    - intent: request_human
    - action: action_handover

- story: user denies human handover
  steps:
    - intent: deny
    - action: utter_recommendations

Overwriting /content/eco_travel_bot/data/stories.yml


## rules

In [16]:
%%writefile /content/eco_travel_bot/data/rules.yml
version: "3.1"

rules:
  - rule: Start trip planning
    steps:
      - intent: plan_trip
      - action: action_check_missing_slots

  - rule: Set destination after asking - provide_destination
    steps:
      - action: action_ask_destination
      - intent: provide_destination
      - action: action_set_destination

  - rule: Set destination after asking - nlu_fallback
    steps:
      - action: action_ask_destination
      - intent: nlu_fallback
      - action: action_set_destination

  - rule: Set destination after asking - out_of_scope
    steps:
      - action: action_ask_destination
      - intent: out_of_scope
      - action: action_set_destination

  - rule: Set destination after asking - provide_origin
    steps:
      - action: action_ask_destination
      - intent: provide_origin
      - action: action_set_destination

  - rule: Set destination after asking - plan_trip
    steps:
      - action: action_ask_destination
      - intent: plan_trip
      - action: action_set_destination

  - rule: Set origin after asking - provide_origin
    steps:
      - action: action_ask_origin
      - intent: provide_origin
      - action: action_set_origin

  - rule: Set origin after asking - provide_destination
    steps:
      - action: action_ask_origin
      - intent: provide_destination
      - action: action_set_origin

  - rule: Set origin after asking - plan_trip
    steps:
      - action: action_ask_origin
      - intent: plan_trip
      - action: action_set_origin

  - rule: Set origin after asking - nlu_fallback
    steps:
      - action: action_ask_origin
      - intent: nlu_fallback
      - action: action_set_origin

  - rule: Set origin after asking - out_of_scope
    steps:
      - action: action_ask_origin
      - intent: out_of_scope
      - action: action_set_origin

  - rule: Set origin via GPS
    steps:
      - action: action_ask_origin
      - intent: provide_location_method
      - action: action_set_origin

  - rule: Provide date
    steps:
      - intent: provide_date
      - action: action_check_missing_slots

  - rule: Provide budget
    steps:
      - intent: provide_budget
      - action: action_check_missing_slots

  - rule: Provide preference
    steps:
      - intent: provide_preference
      - action: action_check_missing_slots

  - rule: Provide transport mode - trip planning flow
    steps:
      - intent: provide_transport_mode
      - action: action_check_missing_slots

  - rule: Provide transport mode - compare flow
    steps:
      - intent: provide_transport_mode_compare
      - action: action_confirm_compare_selection

  - rule: Provide location method
    steps:
      - intent: provide_location_method
      - action: action_check_missing_slots

  - rule: Compare transport options
    steps:
      - intent: compare_transport
      - action: action_compare_transport_options

  - rule: Compare transport - city input as nlu_fallback
    steps:
      - action: action_compare_transport_options
      - intent: nlu_fallback
      - action: action_compare_transport_options

  - rule: Compare transport - city input as out_of_scope
    steps:
      - action: action_compare_transport_options
      - intent: out_of_scope
      - action: action_compare_transport_options

  - rule: Compare transport - city input as plan_trip
    steps:
      - action: action_compare_transport_options
      - intent: plan_trip
      - action: action_compare_transport_options

  - rule: Compare transport - city input as provide_destination
    steps:
      - action: action_compare_transport_options
      - intent: provide_destination
      - action: action_compare_transport_options

  - rule: Show local activities
    steps:
      - intent: show_local_activities
      - action: action_show_local_activities

  - rule: Set selected option
    steps:
      - intent: provide_option
      - action: action_set_selected_option

  - rule: Request human advisor
    steps:
      - intent: request_human
      - action: action_handover

  - rule: Say goodbye
    steps:
      - intent: goodbye
      - action: utter_goodbye

  - rule: Privacy info
    steps:
      - intent: ask_privacy
      - action: utter_privacy_info

  - rule: Accessibility info
    steps:
      - intent: ask_accessibility
      - action: utter_accessibility

  - rule: Ethics info
    steps:
      - intent: ask_ethics
      - action: utter_ethics

  - rule: Fallback
    steps:
      - intent: nlu_fallback
      - action: action_default_fallback

  - rule: Out of scope
    steps:
      - intent: out_of_scope
      - action: utter_out_of_scope

Overwriting /content/eco_travel_bot/data/rules.yml


# action.py

In [17]:
%%writefile /content/eco_travel_bot/actions/__init__.py

Overwriting /content/eco_travel_bot/actions/__init__.py


In [18]:
%%writefile /content/eco_travel_bot/actions/actions.py
from typing import Any, Text, Dict, List
import json
import os
import re
import nltk

from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.chunk import tree2conlltags

from rasa_sdk import Action, Tracker
from rasa_sdk.executor import CollectingDispatcher
from rasa_sdk.events import SlotSet, FollowupAction


_ACTIVITIES_PATH = os.path.join(os.path.dirname(__file__), "..", "data", "local_activities.json")
with open(_ACTIVITIES_PATH, "r", encoding="utf-8") as f:
    CITY_ACTIVITIES = json.load(f)

_TRAVEL_DATA_PATH = os.path.join(os.path.dirname(__file__), "..", "data", "mock_travel_data.json")
with open(_TRAVEL_DATA_PATH, "r", encoding="utf-8") as f:
    TRAVEL_DATA = json.load(f)


try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
except (ImportError, OSError):
    spacy = None
    nlp = None


# ─────────────────────────────────────────────
# NLP YARDIMCI FONKSİYONLAR
# ─────────────────────────────────────────────

def extract_with_spacy(text):
    entities = {
        "destination": None, "origin": None, "date": None,
        "budget": None, "preference": None,
        "transport_mode": None, "location_method": None
    }
    if not text:
        return entities
    lower_text = text.lower()
    if nlp:
        doc = nlp(text)
        locations = []
        for ent in doc.ents:
            if ent.label_ == "GPE":
                locations.append(ent.text)
            elif ent.label_ == "DATE":
                entities["date"] = ent.text
            elif ent.label_ == "MONEY":
                entities["budget"] = ent.text
        orig_r, dest_r = extract_origin_destination(text)
        if orig_r and dest_r:
            entities["origin"]      = orig_r
            entities["destination"] = dest_r
        elif len(locations) == 1:
            entities["destination"] = locations[0]
        elif len(locations) >= 2:
            entities["origin"]      = locations[0]
            entities["destination"] = locations[1]
    budget_match = re.search(r"(\d+)\s*(euro|euros|eur|€|pound|gbp|\$|dollar)?", lower_text)
    if budget_match and not entities["budget"]:
        entities["budget"] = budget_match.group(1) + " euros"
    if "train" in lower_text:
        entities["transport_mode"] = "train"
    elif "bus" in lower_text:
        entities["transport_mode"] = "bus"
    elif "flight" in lower_text or "plane" in lower_text:
        entities["transport_mode"] = "flight"
    elif "car" in lower_text:
        entities["transport_mode"] = "car"
    if "gps" in lower_text or "current location" in lower_text or "detect my location" in lower_text:
        entities["location_method"] = "gps"
    if "eco" in lower_text or "sustainable" in lower_text or "green" in lower_text or "low carbon" in lower_text:
        entities["preference"] = "high"
    elif "balanced" in lower_text:
        entities["preference"] = "medium"
    elif "not sustainable" in lower_text or "do not care" in lower_text or "dont care" in lower_text:
        entities["preference"] = "low"
    return entities


def extract_with_nltk(text):
    result = {"origin": None, "destination": None}
    if not text:
        return result
    try:
        tokens         = word_tokenize(text)
        pos_tags       = pos_tag(tokens)
        named_entities = ne_chunk(pos_tags)
        iob_tags       = tree2conlltags(named_entities)
        locations      = []
        current_entity = []
        for word, pos, ne in iob_tags:
            if ne.startswith("B-GPE"):
                if current_entity:
                    locations.append(" ".join(current_entity))
                    current_entity = []
                current_entity.append(word)
            elif ne.startswith("I-GPE") and current_entity:
                current_entity.append(word)
            elif current_entity and ne == "O":
                locations.append(" ".join(current_entity))
                current_entity = []
        if current_entity:
            locations.append(" ".join(current_entity))
        if len(locations) == 1:
            result["destination"] = locations[0]
        elif len(locations) >= 2:
            result["origin"]      = locations[0]
            result["destination"] = locations[1]
    except:
        pass
    return result


def extract_origin_destination(text):
    """Hem 'from X to Y' hem de 'X to Y' (from'suz) pattern'larini yakalar."""
    if not text:
        return None, None
    stop_words = {"i", "want", "to", "travel", "from", "the", "a", "an",
                  "go", "plan", "trip", "help", "need", "would", "like",
                  "visit", "me", "my", "is", "am", "will", "please",
                  "by", "via", "using", "with", "and", "or", "that", "this"}

    m = re.search(
        r"from\s+([a-zA-Z][a-zA-Z\s]{1,20}?)\s+to\s+([a-zA-Z][a-zA-Z\s]{1,20})"
        r"(?:\s|$|by|via|next|under|with)",
        text, re.IGNORECASE)
    if m:
        orig = m.group(1).strip().title()
        dest = m.group(2).strip().title()
        if orig.lower() not in stop_words and dest.lower() not in stop_words:
            return orig, dest
    words = text.lower().split()
    for idx, word in enumerate(words):
        if word == "to" and idx > 0 and idx < len(words) - 1:
            prev_clean = re.sub(r'[^a-z]', '', words[idx - 1])
            next_clean = re.sub(r'[^a-z]', '', words[idx + 1])
            if (prev_clean not in stop_words and len(prev_clean) >= 3 and
                    next_clean not in stop_words and len(next_clean) >= 3):
                return prev_clean.title(), next_clean.title()
    return None, None


def combined_nlp_extraction(text):
    orig_regex, dest_regex = extract_origin_destination(text)
    spacy_result = extract_with_spacy(text)
    nltk_result  = extract_with_nltk(text)
    if orig_regex:
        spacy_result["origin"]      = orig_regex
    if dest_regex:
        spacy_result["destination"] = dest_regex
    if not spacy_result.get("origin"):
        spacy_result["origin"]      = nltk_result.get("origin")
    if not spacy_result.get("destination"):
        spacy_result["destination"] = nltk_result.get("destination")
    return spacy_result


def extract_any_city(text):
    if not text:
        return None
    text = text.strip()
    if text.startswith("/"):
        return None

    result = combined_nlp_extraction(text)
    city = result.get("destination") or result.get("origin")
    if city:
        return city

    _, dest_r = extract_origin_destination(text)
    if dest_r:
        return dest_r

    to_city = re.search(
        r"(?:travel|go|visit|trip)\s+to\s+([a-zA-Z]{3,20})\b",
        text, re.IGNORECASE)
    if to_city:
        return to_city.group(1).title()

    prep_city = re.search(
        r"^(?:to|from)\s+([A-Za-zÀ-ÖØ-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ\s\-]{1,30})$",
        text, re.IGNORECASE)
    if prep_city:
        return prep_city.group(1).strip().title()

    words = text.strip().split()
    stop_words = {"i", "want", "to", "travel", "from", "the", "a", "an",
                  "go", "plan", "trip", "help", "need", "would", "like",
                  "visit", "me", "my", "is", "am", "will", "please"}

    if len(words) == 2 and words[0].lower() in {"to", "from"}:
        candidate = words[1]
        if re.match(r"^[A-Za-zÀ-ÖØ-öø-ÿ\-]{2,40}$", candidate):
            return candidate.title()

    if len(words) <= 3 and not any(w.lower() in stop_words for w in words):
        if re.match(r"^[A-Za-zÀ-ÖØ-öø-ÿ\s\-]{2,40}$", text):
            return text.title()

    return None


# ─────────────────────────────────────────────
# YARDIMCI FONKSİYONLAR
# ─────────────────────────────────────────────

def clean_budget_value(budget):
    if not budget:
        return None
    t = str(budget).lower()
    if "low" in t or "cheap" in t or "budget" in t:   return 300
    if "medium" in t or "balanced" in t:               return 600
    if "high" in t or "flexible" in t:                 return 900
    digits = "".join(c for c in t if c.isdigit())
    return int(digits) if digits else None


def map_budget_to_tier(budget_value):
    """Sayısal bütçeyi tier string'ine çevirir."""
    if not budget_value:    return "mid"
    if budget_value <= 450: return "budget"
    if budget_value <= 750: return "mid"
    return "high"


def normalize_preference(preference):
    if not preference:
        return "medium"
    p = str(preference).lower()
    if any(w in p for w in ["high", "eco-friendly", "eco friendly", "low carbon", "sustainable", "green", "most eco"]):
        return "high"
    if any(w in p for w in ["low", "cheap", "low-cost", "low cost", "budget"]):
        return "low"
    return "medium"


def get_emission_label(carbon_kg):
    if carbon_kg <= 60:  return "Green / Low emission"
    if carbon_kg <= 130: return "Amber / Moderate emission"
    return "Red / High emission"


def get_score_label(score):
    if score >= 90: return "Green / Eco-friendly"
    if score >= 60: return "Amber / Moderate impact"
    return "Red / High emission"


def calculate_weighted_score(option, user_budget, preference):
    carbon_score = max(0, 100 - option["carbon_kg"])
    price_score  = 100
    if user_budget:
        price_score = max(0, 100 - abs(option["price"] - user_budget) / user_budget * 100)
    if preference == "high":
        return round(0.55 * carbon_score + 0.20 * price_score + 0.25 * option["sustainability_score"], 2)
    if preference == "low":
        return round(0.25 * carbon_score + 0.50 * price_score + 0.25 * option["sustainability_score"], 2)
    return round(0.40 * carbon_score + 0.35 * price_score + 0.25 * option["sustainability_score"], 2)


def extracted_nlp_slot(slot_name, text):
    if not text:
        return None
    lower = text.lower()

    if slot_name == "transport_mode":
        for mode in ["train", "bus", "flight", "car"]:
            if mode in lower:
                return mode
        if "plane" in lower:
            return "flight"

    elif slot_name == "date":
        date_map = {
            "next week":    ["next week", "newt week", "nxt week", "next wek"],
            "tomorrow":     ["tomorrow", "tommorow", "tomorow"],
            "today":        ["today", "toady"],
            "next month":   ["next month", "nxt month"],
            "this weekend": ["this weekend", "this weakend"],
        }
        for canonical, variants in date_map.items():
            if any(v in lower for v in variants):
                return canonical
        for kw in ["today", "tomorrow", "next week", "next month", "this weekend", "next saturday"]:
            if kw in lower:
                return kw
        m = re.search(
            r"(january|february|march|april|may|june|july|august|september|october|november|december"
            r"|jan|feb|mar|apr|jun|jul|aug|sep|oct|nov|dec)\s+\d{1,2}", lower)
        if m:
            return m.group(0)

    elif slot_name == "budget":
        m = re.search(r"(\d+)\s*(euro|euros|eur|€|pound|gbp|\$|dollar)?", lower)
        if m:
            return m.group(1) + " euros"

    elif slot_name == "location_method":
        if "gps" in lower or "current location" in lower or "detect" in lower:
            return "gps"

    elif slot_name == "preference":
        if any(w in lower for w in ["eco-friendly", "eco friendly", "ecofriendly",
                                     "eco", "sustainable", "green", "low carbon"]):
            return "high"
        elif "balanced" in lower:
            return "medium"
        elif any(w in lower for w in ["not sustainable", "do not care", "dont care", "cheap"]):
            return "low"

    return None


# ─────────────────────────────────────────────
# CONTEXT-AWARE ŞEHİR ÇIKARMA
# ─────────────────────────────────────────────

STOP_WORDS = {"i", "want", "to", "travel", "from", "the", "a", "an",
              "go", "plan", "trip", "help", "need", "would", "like",
              "use", "gps", "my", "please", "is", "am", "will"}


def extract_city_in_context(text, rasa_entities, slot_key):
    if not text:
        return None

    text_stripped = text.strip()

    city = rasa_entities.get(slot_key)
    if city:
        return city

    if slot_key == "origin":
        m = re.match(
            r"^(?:from|i(?:'m|'m)?\s+(?:from|coming from|traveling from|travelling from))\s+"
            r"([A-Za-zÀ-ÖØ-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ\s\-]{1,30})$",
            text_stripped, re.IGNORECASE)
        if m:
            candidate = m.group(1).strip().title()
            if candidate.lower() not in STOP_WORDS and len(candidate) >= 2:
                return candidate

        m2 = re.search(
            r"(?:^|\s)from\s+([A-Za-zÀ-ÖØ-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ\s\-]{1,30})"
            r"(?:\s*$|\s+(?:by|to|via|next|under|with))",
            text_stripped, re.IGNORECASE)
        if m2:
            candidate = m2.group(1).strip().title()
            if candidate.lower() not in STOP_WORDS and len(candidate) >= 2:
                return candidate

    elif slot_key == "destination":
        m = re.match(
            r"^(?:to|i(?:'m|'m)?\s+(?:going to|traveling to|travelling to|want to go to|would like to visit))\s+"
            r"([A-Za-zÀ-ÖØ-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ\s\-]{1,30})$",
            text_stripped, re.IGNORECASE)
        if m:
            candidate = m.group(1).strip().title()
            if candidate.lower() not in STOP_WORDS and len(candidate) >= 2:
                return candidate

        m2 = re.search(
            r"(?:travel|go|visit|trip)\s+to\s+([A-Za-zÀ-ÖØ-öø-ÿ]{3,20})\b",
            text_stripped, re.IGNORECASE)
        if m2:
            return m2.group(1).title()

    words = text_stripped.split()
    if 1 <= len(words) <= 3:
        non_stop = [w for w in words if w.lower() not in STOP_WORDS]
        if non_stop and all(re.match(r"^[A-Za-zÀ-ÖØ-öø-ÿ\-]{2,}$", w) for w in non_stop):
            return " ".join(non_stop).title()

    other_key = "destination" if slot_key == "origin" else "origin"
    city = rasa_entities.get(other_key) or extract_any_city(text_stripped)
    return city


# ─────────────────────────────────────────────
# ACTION: DESTINATION SORMA VE SET ETME
# ─────────────────────────────────────────────

class ActionAskDestination(Action):
    def name(self): return "action_ask_destination"

    def run(self, dispatcher, tracker, domain):
        dispatcher.utter_message(text="Where would you like to travel? Please type your destination city.")
        return [SlotSet("requested_slot", "destination")]


class ActionSetDestination(Action):
    def name(self): return "action_set_destination"

    def run(self, dispatcher, tracker, domain):
        rasa_entities = {e["entity"]: e["value"]
                         for e in tracker.latest_message.get("entities", [])}
        latest_text   = tracker.latest_message.get("text") or ""

        city = extract_city_in_context(latest_text, rasa_entities, "destination")

        if city:
            return [
                SlotSet("destination", city),
                SlotSet("requested_slot", None),
                FollowupAction("action_check_missing_slots")
            ]

        dispatcher.utter_message(
            text="I could not recognise a city name. Please type the destination city, e.g. 'Paris' or 'Berlin'.")
        return []


# ─────────────────────────────────────────────
# ACTION: ORIGIN SORMA VE SET ETME
# ─────────────────────────────────────────────

class ActionAskOrigin(Action):
    def name(self): return "action_ask_origin"

    def run(self, dispatcher, tracker, domain):
        dispatcher.utter_message(
            text="Where are you traveling from? Type your departure city or use GPS.",
            buttons=[{"title": "Use GPS",
                      "payload": '/provide_location_method{"location_method":"gps"}'}])
        return [SlotSet("requested_slot", "origin")]


class ActionSetOrigin(Action):
    def name(self): return "action_set_origin"

    def run(self, dispatcher, tracker, domain):
        rasa_entities = {e["entity"]: e["value"]
                         for e in tracker.latest_message.get("entities", [])}
        latest_text   = tracker.latest_message.get("text") or ""

        loc_method = rasa_entities.get("location_method", "")
        if str(loc_method).lower() == "gps":
            dispatcher.utter_message(
                text="GPS location retrieved successfully using mock location service.")
            return [
                SlotSet("origin", "GPS current location"),
                SlotSet("requested_slot", None),
                FollowupAction("action_check_missing_slots")
            ]

        city = extract_city_in_context(latest_text, rasa_entities, "origin")

        if city:
            return [
                SlotSet("origin", city),
                SlotSet("requested_slot", None),
                FollowupAction("action_check_missing_slots")
            ]

        dispatcher.utter_message(
            text="I could not recognise a city name. Please type your departure city, e.g. 'Berlin'.")
        return []


# ─────────────────────────────────────────────
# ACTION: EKSİK SLOT KONTROLÜ (ANA ORKESTRATÖR)
# ─────────────────────────────────────────────

class ActionCheckMissingSlots(Action):
    def name(self): return "action_check_missing_slots"

    def run(self, dispatcher, tracker, domain):
        dest            = tracker.get_slot("destination")
        origin          = tracker.get_slot("origin")
        date            = tracker.get_slot("date")
        budget          = tracker.get_slot("budget")
        preference      = tracker.get_slot("preference")
        location_method = tracker.get_slot("location_method")
        transport_mode  = tracker.get_slot("transport_mode")

        latest_text    = tracker.latest_message.get("text") or ""
        intent         = tracker.latest_message.get("intent", {}).get("name", "")
        rasa_entities  = {e["entity"]: e["value"]
                          for e in tracker.latest_message.get("entities", [])}

        requested_slot = tracker.get_slot("requested_slot") or ""

        events = []

        if not dest:
            if requested_slot == "destination":
                city = extract_city_in_context(latest_text, rasa_entities, "destination")
                if city:
                    dest = city
                    events.append(SlotSet("destination", city))
                    events.append(SlotSet("requested_slot", None))
            elif intent in {"plan_trip", "provide_destination"}:
                city = (rasa_entities.get("destination")
                        or extract_any_city(latest_text))
                if city:
                    dest = city
                    events.append(SlotSet("destination", city))

        if not origin:
            if requested_slot == "origin":
                loc_method = rasa_entities.get("location_method", "")
                if str(loc_method).lower() == "gps":
                    dispatcher.utter_message(
                        text="GPS location retrieved successfully using mock location service.")
                    origin = "GPS current location"
                    events.append(SlotSet("origin", origin))
                    events.append(SlotSet("requested_slot", None))
                else:
                    city = extract_city_in_context(latest_text, rasa_entities, "origin")
                    if city:
                        origin = city
                        events.append(SlotSet("origin", city))
                        events.append(SlotSet("requested_slot", None))
            elif intent in {"plan_trip", "provide_origin"}:
                _nlp = combined_nlp_extraction(latest_text) if latest_text else {}
                city = (rasa_entities.get("origin") or _nlp.get("origin"))
                if city:
                    origin = city
                    events.append(SlotSet("origin", city))

        other_slots = ["date", "budget", "preference", "transport_mode", "location_method"]
        for slot_name in other_slots:
            if not tracker.get_slot(slot_name):
                val = rasa_entities.get(slot_name) or extracted_nlp_slot(slot_name, latest_text)
                if val:
                    events.append(SlotSet(slot_name, val))
                    if   slot_name == "date":            date            = val
                    elif slot_name == "budget":          budget          = val
                    elif slot_name == "preference":      preference      = val
                    elif slot_name == "transport_mode":  transport_mode  = val
                    elif slot_name == "location_method": location_method = val

        if not dest:
            events.append(FollowupAction("action_ask_destination"))
            return events

        if not origin:
            if str(location_method or "").lower() == "gps":
                dispatcher.utter_message(
                    text="GPS location retrieved successfully using mock location service.")
                events.append(SlotSet("origin", "GPS current location"))
                events.append(FollowupAction("action_check_missing_slots"))
                return events
            events.append(FollowupAction("action_ask_origin"))
            return events

        if not date:
            dispatcher.utter_message(response="utter_ask_date")
            return events

        if not budget:
            dispatcher.utter_message(response="utter_ask_budget")
            return events

        if not preference or preference not in ["low", "medium", "high"]:
            dispatcher.utter_message(response="utter_ask_preference")
            return events

        if not transport_mode:
            dispatcher.utter_message(response="utter_ask_transport_mode")
            return events

        dispatcher.utter_message(
            text="All required travel details are collected. I will now search for sustainable options.")
        events.append(FollowupAction("action_call_travel_api"))
        return events


# ─────────────────────────────────────────────
# ACTION: TRANSPORT KARŞILAŞTIRMA
# ─────────────────────────────────────────────

class ActionCompareTransportOptions(Action):
    def name(self): return "action_compare_transport_options"

    def run(self, dispatcher, tracker, domain):
        origin      = tracker.get_slot("origin")
        destination = tracker.get_slot("destination")
        latest_text = tracker.latest_message.get("text") or ""
        extracted   = combined_nlp_extraction(latest_text)
        events      = []

        if not origin and extracted.get("origin"):
            origin = extracted["origin"]
            events.append(SlotSet("origin", origin))
        if not destination and extracted.get("destination"):
            destination = extracted["destination"]
            events.append(SlotSet("destination", destination))

        if not origin or not destination:
            dispatcher.utter_message(
                text="Sure 🌱 I can compare transport options. Please tell me your origin and destination, e.g. 'Berlin to Paris'.",
                buttons=[
                    {"title": "Plan full trip",        "payload": "/plan_trip"},
                    {"title": "Talk to human advisor",  "payload": "/request_human"},
                ])
            return events

        profiles = TRAVEL_DATA.get("transport_carbon_profiles", {})
        options = [
            {
                "mode":        k.title(),
                "eco_score":   v["eco_score"],
                "price":       v["price_tier"].title(),
                "status":      v["carbon_label"],
                "description": v["description"],
            }
            for k, v in profiles.items()
        ]
        options.sort(key=lambda x: x["eco_score"], reverse=True)

        msg = f"Here are the transport options for {origin} to {destination}:\n\n"
        for o in options:
            msg += (f"Mode: {o['mode']}\n"
                    f"Eco Score: {o['eco_score']}/100\n"
                    f"Price tier: {o['price']}\n"
                    f"Status: {o['status']}\n"
                    f"Note: {o['description']}\n\n")
        msg += "A score above 80 is considered eco-friendly."

        dispatcher.utter_message(text=msg, buttons=[
            {"title": "Choose Train",          "payload": '/provide_transport_mode_compare{"transport_mode":"train"}'},
            {"title": "Choose Bus",            "payload": '/provide_transport_mode_compare{"transport_mode":"bus"}'},
            {"title": "Choose Flight",         "payload": '/provide_transport_mode_compare{"transport_mode":"flight"}'},
            {"title": "Choose Car",            "payload": '/provide_transport_mode_compare{"transport_mode":"car"}'},
            {"title": "Talk to human advisor", "payload": "/request_human"},
        ])
        return events


# ─────────────────────────────────────────────
# ACTION: COMPARE SONRASI TRANSPORT SEÇİMİ
# ─────────────────────────────────────────────

class ActionConfirmCompareSelection(Action):
    def name(self): return "action_confirm_compare_selection"

    def run(self, dispatcher, tracker, domain):
        rasa_entities  = {e["entity"]: e["value"]
                          for e in tracker.latest_message.get("entities", [])}
        transport_mode = rasa_entities.get("transport_mode") or tracker.get_slot("transport_mode")
        origin         = tracker.get_slot("origin") or "your origin"
        destination    = tracker.get_slot("destination") or "your destination"

        eco_notes = {
            "train":  "🟢 Most eco-friendly choice!",
            "bus":    "🟢 Great eco-friendly choice!",
            "car":    "🟡 Moderate carbon footprint.",
            "flight": "🔴 Highest carbon footprint.",
        }
        note = eco_notes.get(str(transport_mode).lower(), "")

        dispatcher.utter_message(
            text=(f"You selected {str(transport_mode).title()} from {origin} to {destination}. {note}\n\n"
                  f"Would you like to plan a full trip with this transport option?"),
            buttons=[
                {"title": "Yes, plan my trip 🗺️",    "payload": "/plan_trip"},
                {"title": "Talk to human advisor 👤", "payload": "/request_human"},
                {"title": "No, end conversation",     "payload": "/deny"},
            ]
        )
        return [
            SlotSet("transport_mode", transport_mode),
            SlotSet("origin", origin),       # ← ekle
            SlotSet("destination", destination),  # ← ekle
        ]


# ─────────────────────────────────────────────
# ACTION: TRAVEL API + CARBON + RANKING
# ─────────────────────────────────────────────

class ActionCallTravelAPI(Action):
    def name(self): return "action_call_travel_api"

    def run(self, dispatcher, tracker, domain):
        origin      = tracker.get_slot("origin")
        destination = tracker.get_slot("destination")
        date        = tracker.get_slot("date")
        budget      = tracker.get_slot("budget")
        preference  = normalize_preference(tracker.get_slot("preference"))
        try:
            dispatcher.utter_message(
                text=f"Searching travel options from {origin} to {destination} for {date}, budget {budget}, sustainability {preference}.")
            dispatcher.utter_message(text="Live API not connected. Using mock fallback data.")
            return [FollowupAction("action_calculate_carbon")]
        except Exception:
            cnt = tracker.get_slot("api_error_count") or 0
            dispatcher.utter_message(text="Travel data could not be retrieved. Trying again or escalating.")
            return [SlotSet("api_error_count", cnt + 1), FollowupAction("action_retry_or_escalate")]


class ActionCalculateCarbon(Action):
    def name(self): return "action_calculate_carbon"

    def run(self, dispatcher, tracker, domain):
        try:
            dispatcher.utter_message(text="Climatiq API not connected. Using mock carbon values.")
            return [FollowupAction("action_rank_options")]
        except Exception:
            cnt = tracker.get_slot("api_error_count") or 0
            dispatcher.utter_message(text="Carbon calculation failed.")
            return [SlotSet("api_error_count", cnt + 1), FollowupAction("action_retry_or_escalate")]


class ActionRankOptions(Action):
    def name(self): return "action_rank_options"

    def run(self, dispatcher, tracker, domain):
        budget         = clean_budget_value(tracker.get_slot("budget"))
        preference     = normalize_preference(tracker.get_slot("preference"))
        transport_mode = tracker.get_slot("transport_mode")
        destination    = tracker.get_slot("destination") or ""

        try:
            # ── mock_travel_data.json'dan şehir + tier bazlı seçenekleri çek ──
            city_key  = destination.lower().strip()
            tier      = map_budget_to_tier(budget)
            city_data = TRAVEL_DATA["travel_options"].get(city_key, {})
            options   = list(city_data.get(tier, []))

            # Tier'da seçenek yoksa tüm tier'ları birleştir (fallback)
            if not options:
                for t in ["budget", "mid", "high"]:
                    options += city_data.get(t, [])

            # Hiç seçenek yoksa (bilinmeyen şehir) tüm şehirlerden rastgele al
            if not options:
                for c_data in TRAVEL_DATA["travel_options"].values():
                    for t in ["budget", "mid", "high"]:
                        options += c_data.get(t, [])
                    if len(options) >= 8:
                        break

            # Transport filtresi
            if transport_mode and str(transport_mode).lower() not in ["any", "no preference", "no_preference"]:
                  mode = str(transport_mode).lower()
                  filtered = [o for o in options if o["transport"] == mode]
                  if len(filtered) >= 2:
                      options = filtered
                  elif len(filtered) == 1:
                      all_city = []
                      for t in ["budget", "mid", "high"]:
                          all_city += city_data.get(t, [])
                      filtered = [o for o in all_city if o["transport"] == mode]
                      options = filtered if filtered else options

            # Skor hesapla ve sırala
            for o in options:
                o["final_score"]    = calculate_weighted_score(o, budget, preference)
                o["emission_label"] = get_emission_label(o["carbon_kg"])
                o["score_label"]    = get_score_label(o["final_score"])

            ranked = sorted(options, key=lambda x: x["final_score"], reverse=True)[:4]

            resp = "Here are the ranked sustainable travel options:\n\n"
            for i, o in enumerate(ranked, 1):
                eco_badge = "✅ Eco-certified" if o.get("eco_certified") else "⬜ Standard"
                resp += (f"{i}. {o['transport'].title()} + {o['hotel']}\n"
                         f"Price: €{o['price']}  |  Carbon: {o['carbon_kg']} kg CO₂\n"
                         f"Sustainability: {o['sustainability_score']}/100  |  {eco_badge}\n"
                         f"Score: {o['final_score']}/100  |  {o['emission_label']}\n\n")

            dispatcher.utter_message(text=resp)
            buttons = []
            for i, o in enumerate(ranked, 1):
                label = f"{i}. {o['transport'].title()} + {o['hotel']} (€{o['price']})"
                buttons.append({
                    "title":   label,
                    "payload": f'/provide_option{{"selected_option":"{i}"}}'
                })
            dispatcher.utter_message(
                text="Which option would you like to select? You can tap a button or type the option number.",
                buttons=buttons
            )
            return []
        except Exception:
            dispatcher.utter_message(text="Could not rank options. Escalating.")
            return [SlotSet("handover_reason", "ranking_error"), FollowupAction("action_handover")]


class ActionSetSelectedOption(Action):
    def name(self): return "action_set_selected_option"

    def run(self, dispatcher, tracker, domain):
        rasa_entities = {e["entity"]: e["value"]
                         for e in tracker.latest_message.get("entities", [])}

        selected = rasa_entities.get("selected_option")
        if not selected:
            text = tracker.latest_message.get("text", "")
            match = re.search(r"\b([1-4])\b", text)
            if match:
                selected = match.group(1)

        if not selected:
            dispatcher.utter_message(
                text="I did not catch that. Please type the option number (1-4) or tap a button."
            )
            return []

        dispatcher.utter_message(
            text=f"Got it! You selected option {selected}. Would you like to see local activities in your destination?",
            buttons=[
                {"title": "Yes, show local activities", "payload": "/affirm"},
                {"title": "No, thanks",                 "payload": "/deny"},
            ]
        )
        return [SlotSet("selected_option", selected)]


# ─────────────────────────────────────────────
# ACTION: RETRY / ESCALATE / HANDOVER / FALLBACK
# ─────────────────────────────────────────────

class ActionRetryOrEscalate(Action):
    def name(self): return "action_retry_or_escalate"

    def run(self, dispatcher, tracker, domain):
        clarification_count = tracker.get_slot("clarification_count") or 0
        api_error_count     = tracker.get_slot("api_error_count") or 0
        if api_error_count >= 2:
            dispatcher.utter_message(text="Repeated API errors. Escalating to human advisor.")
            return [SlotSet("handover_reason", "repeated_api_error"), FollowupAction("action_handover")]
        if clarification_count >= 2:
            dispatcher.utter_message(text="Still unclear. Connecting to human advisor.")
            return [SlotSet("handover_reason", "repeated_fallback"), FollowupAction("action_handover")]
        dispatcher.utter_message(
            text="I did not understand clearly. Please choose one option:",
            buttons=[
                {"title": "Plan a trip",               "payload": "/plan_trip"},
                {"title": "Compare transport options",  "payload": "/compare_transport"},
                {"title": "Human advisor",              "payload": "/request_human"},
            ])
        return [SlotSet("clarification_count", clarification_count + 1)]


class ActionHandover(Action):
    def name(self): return "action_handover"

    def run(self, dispatcher, tracker, domain):
        context = {
            "origin":          tracker.get_slot("origin"),
            "destination":     tracker.get_slot("destination"),
            "date":            tracker.get_slot("date"),
            "budget":          tracker.get_slot("budget"),
            "preference":      normalize_preference(tracker.get_slot("preference")),
            "transport_mode":  tracker.get_slot("transport_mode"),
            "location_method": tracker.get_slot("location_method"),
            "handover_reason": tracker.get_slot("handover_reason"),
        }
        dispatcher.utter_message(
            text="Escalated to human advisor.\n\nContext (JSON):\n\n" + json.dumps(context, indent=2))
        return []


class ActionDefaultFallback(Action):
    def name(self): return "action_default_fallback"

    def run(self, dispatcher, tracker, domain):
        return [FollowupAction("action_retry_or_escalate")]


# ─────────────────────────────────────────────
# ACTION: LOCAL AKTİVİTELER
# ─────────────────────────────────────────────

CITY_ALIASES = {
    "roma": "rome",
    "parigi": "paris",
    "berlino": "berlin",
    "londra": "london",
    "wien": "vienna",
    "barcellona": "barcelona",
    "copenhague": "copenhagen",
    "lisbona": "lisbon",
    "varsavia": "warsaw",
    "budapes": "budapest",
}

class ActionShowLocalActivities(Action):
    def name(self): return "action_show_local_activities"

    def run(self, dispatcher, tracker, domain):
        destination   = tracker.get_slot("destination") or "your destination"
        key           = destination.lower().strip()
        key           = CITY_ALIASES.get(key, key)
        preference    = normalize_preference(tracker.get_slot("preference"))
        city_data     = CITY_ACTIVITIES.get(key, CITY_ACTIVITIES["default"])
        activity_list = city_data.get(preference, city_data["medium"])

        msg = f"Here are local activities for {destination}:\n\n"
        for i, activity in enumerate(activity_list, 1):
            msg += f"{i}. {activity}\n\n"

        dispatcher.utter_message(text=msg, buttons=[
            {"title": "Plan a new trip",           "payload": "/plan_trip"},
            {"title": "End Plan",                  "payload": "/deny"},
            {"title": "Talk to human advisor",     "payload": "/request_human"},
        ])
        return []

Overwriting /content/eco_travel_bot/actions/actions.py


# endpoitn.yml

In [19]:
%%writefile /content/eco_travel_bot/endpoints.yml
action_endpoint:
  url: "http://localhost:5055/webhook"


Overwriting /content/eco_travel_bot/endpoints.yml


# Rasa train

In [20]:
import subprocess
result = subprocess.run(
    "source /usr/local/rasa_venv/bin/activate && python /content/eco_travel_bot/generate_mock_data.py",
    shell=True, executable="/bin/bash", capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

✓ mock_travel_data.json generated
  Cities  : 19
  Records : 228 total options across all cities and tiers
  berlin: budget=4 mid=4 high=4
  hamburg: budget=4 mid=4 high=4
  munich: budget=4 mid=4 high=4
  ...




In [21]:
import subprocess, os

os.chdir('/content/eco_travel_bot')

result = subprocess.run(
    'source /usr/local/rasa_venv/bin/activate && rasa train --force 2>&1',
    shell=True,
    executable='/bin/bash',
    capture_output=True,
    text=True
)

# Son 100 satırı göster — asıl hata burda
output = result.stdout + result.stderr
lines = output.strip().split('\n')
print('\n'.join(lines[-100:]))
print(f'\nExit code: {result.returncode}')

Epochs: 100%|██████████| 100/100 [04:19<00:00,  2.59s/it, t_loss=1.89, i_acc=0.986, e_f1=0.969]
2026-05-25 20:42:07 INFO     rasa.engine.training.hooks  - Finished training component 'DIETClassifier'.
2026-05-25 20:42:07 INFO     rasa.engine.training.hooks  - Starting to train component 'EntitySynonymMapper'.
2026-05-25 20:42:07 INFO     rasa.engine.training.hooks  - Finished training component 'EntitySynonymMapper'.
2026-05-25 20:42:07 INFO     rasa.engine.training.hooks  - Starting to train component 'RegexEntityExtractor'.
2026-05-25 20:42:07 INFO     rasa.engine.training.hooks  - Finished training component 'RegexEntityExtractor'.
Your Rasa model is trained and saved at 'models/20260525-203430-proud-circuit.tar.gz'.

Exit code: 0


In [27]:
import subprocess, time

action_proc = subprocess.Popen(
    "source /usr/local/rasa_venv/bin/activate && cd /content/eco_travel_bot && rasa run actions --port 5055",
    shell=True, executable="/bin/bash"
)
time.sleep(8)
print("Action server started on port 5055.")

Action server started on port 5055.


In [28]:
import subprocess, time, requests

rasa_proc = subprocess.Popen(
    'source /usr/local/rasa_venv/bin/activate && '
    'cd /content/eco_travel_bot && '
    'rasa run --enable-api --cors "*" --port 5005',
    shell=True, executable='/bin/bash'
)


# UI ngrok streamlit

In [24]:
!pip install streamlit pyngrok -q

In [25]:
%%writefile /content/eco_travel_bot/streamlit_app.py
import streamlit as st
import requests
import uuid

RASA_URL = "http://localhost:5005/webhooks/rest/webhook"

st.set_page_config(page_title="Eco-Travel Advisor", page_icon="🌿", layout="centered")
st.title("🌿 Eco-Travel Advisor")
st.caption("Sustainable travel planning chatbot")

# Session state başlat
if "messages" not in st.session_state:
    st.session_state.messages = []
    st.session_state.started = False
    st.session_state.sender_id = str(uuid.uuid4())

# İlk açılışta welcome mesajı
if not st.session_state.started:
    st.session_state.messages.append({
        "role": "assistant",
        "content": "Hello! Welcome to Eco-Travel Advisor 🌿 How would you like to start?",
        "buttons": [
            {"title": "Plan a sustainable trip", "payload": "/plan_trip"},
            {"title": "Compare transport options", "payload": "/compare_transport"},
            {"title": "Talk to a human advisor", "payload": "/request_human"}
        ]
    })
    st.session_state.started = True

# Mesajları göster
for i, msg in enumerate(st.session_state.messages):
    with st.chat_message(msg["role"]):
        st.write(msg["content"])
        if msg.get("buttons") and i == len(st.session_state.messages) - 1:
            cols = st.columns(len(msg["buttons"]))
            for j, btn in enumerate(msg["buttons"]):
                if cols[j].button(btn["title"], key=f"btn_{i}_{j}"):
                    st.session_state.pending = {"payload": btn["payload"], "label": btn["title"]}
                    st.rerun()

# Buton tıklaması
if "pending" in st.session_state:
    p = st.session_state.pop("pending")
    st.session_state.messages.append({"role": "user", "content": p["label"], "buttons": []})
    try:
        resp = requests.post(
            RASA_URL,
            json={"sender": st.session_state.sender_id, "message": p["payload"]},
            timeout=10
        )
        for m in resp.json():
            if m.get("text"):
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": m["text"],
                    "buttons": m.get("buttons", [])
                })
    except Exception as e:
        st.session_state.messages.append({"role": "assistant", "content": f"Bağlantı hatası: {e}", "buttons": []})
    st.rerun()

# Metin girişi
user_input = st.chat_input("Type your message...")
if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input, "buttons": []})
    try:
        resp = requests.post(
            RASA_URL,
            json={"sender": st.session_state.sender_id, "message": user_input},
            timeout=10
        )
        for m in resp.json():
            if m.get("text"):
                st.session_state.messages.append({
                    "role": "assistant",
                    "content": m["text"],
                    "buttons": m.get("buttons", [])
                })
    except Exception as e:
        st.session_state.messages.append({"role": "assistant", "content": f"Bağlantı hatası: {e}", "buttons": []})
    st.rerun()

# Reset
if st.button("🔄 Reset Chat"):
    st.session_state.messages = []
    st.session_state.started = False
    st.session_state.sender_id = str(uuid.uuid4())
    st.rerun()

Overwriting /content/eco_travel_bot/streamlit_app.py


In [26]:
from pyngrok import ngrok
from google.colab import userdata
import subprocess, threading, time, sys

def run_streamlit():
    subprocess.Popen([
        sys.executable, "-m", "streamlit", "run",
        "/content/eco_travel_bot/streamlit_app.py",
        "--server.port=8501",
        "--server.headless=true"
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(5)

# AUTH_TOKEN burada kullanılıyor — Colab Secrets'tan alıyor
ngrok.kill()
ngrok.set_auth_token(userdata.get('NGROK'))  # Secrets'taki key adı: NGROK

public_url = ngrok.connect(addr="8501", bind_tls=True)
print(f"🌿 Eco-Travel Advisor hazır!")
print(f"🔗 URL: {public_url}")

🌿 Eco-Travel Advisor hazır!
🔗 URL: NgrokTunnel: "https://parsnip-handclap-trolling.ngrok-free.dev" -> "http://localhost:8501"
